<a href="https://colab.research.google.com/github/njalagam23/neural-networks-from-scratch/blob/main/day11_uvicorn_http_requests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 11 — Serving a PyTorch Model with FastAPI and Uvicorn

## Objective
Run our existing customer churn prediction API as a live HTTP server.

## Business Scenario
A telecommunications company's retention team needs to submit
customer information and receive churn-risk predictions.

## Technologies
Python, PyTorch, FastAPI, Uvicorn, Requests, Pytest

In [14]:
from pathlib import Path
import subprocess

repo = Path("/content/churn-repo")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/njalagam23/neural-networks-from-scratch.git",
            str(repo)
        ],
        check=True
    )

print("Repository cloned:", repo.exists())

Repository cloned: True


In [15]:
project = repo / "customer-churn-app"

required_files = [
    "api.py",
    "app/model.py",
    "app/preprocessing.py",
    "app/predict.py",
    "artifacts/churn_model.pth",
    "artifacts/churn_scaler.joblib",
    "artifacts/model_config.json"
]

for file in required_files:
    path = project / file
    print(f"{file}: {path.exists()}")

api.py: True
app/model.py: True
app/preprocessing.py: True
app/predict.py: True
artifacts/churn_model.pth: True
artifacts/churn_scaler.joblib: True
artifacts/model_config.json: True


In [16]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel, Field
from app.predict import ChurnPredictor

app = FastAPI(
    title="Customer Churn Prediction API",
    version="1.0.0"
)

predictor = ChurnPredictor()


class CustomerInput(BaseModel):
    usage: float = Field(ge=0)
    tenure: int = Field(ge=0)
    tickets: int = Field(ge=0)


@app.get("/")
def home():
    return {"message": "Churn Prediction API is running"}


@app.post("/predict")
def predict_churn(customer: CustomerInput):
    return predictor.predict(
        usage=customer.usage,
        tenure=customer.tenure,
        tickets=customer.tickets
    )
'''

(project / "api.py").write_text(api_code)

print("api.py created:", (project / "api.py").exists())

api.py created: True


In [17]:
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

project.mkdir(parents=True, exist_ok=True)

requirements = """torch
numpy
scikit-learn
joblib
fastapi
uvicorn
requests
httpx
pytest
"""

(project / "requirements.txt").write_text(requirements)

print((project / "requirements.txt").read_text())

torch
numpy
scikit-learn
joblib
fastapi
uvicorn
requests
httpx
pytest



In [18]:
%pip install -q -r /content/churn-repo/customer-churn-app/requirements.txt

In [19]:
import torch
import fastapi
import uvicorn
import requests

print("PyTorch:", torch.__version__)
print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)
print("Requests:", requests.__version__)

print("All dependencies installed successfully!")

PyTorch: 2.11.0+cpu
FastAPI: 0.141.1
Uvicorn: 0.54.0
Requests: 2.32.4
All dependencies installed successfully!


In [20]:
required_files = [
    "api.py",
    "app/model.py",
    "app/preprocessing.py",
    "app/predict.py",
    "artifacts/churn_model.pth",
    "artifacts/churn_scaler.joblib",
    "artifacts/model_config.json",
    "requirements.txt"
]

for file in required_files:
    print(file, ":", (project / file).exists())

api.py : True
app/model.py : True
app/preprocessing.py : True
app/predict.py : True
artifacts/churn_model.pth : True
artifacts/churn_scaler.joblib : True
artifacts/model_config.json : True
requirements.txt : True


In [21]:
import subprocess
import sys
import time
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

# Save server output so we can troubleshoot errors
log_file = project / "uvicorn.log"

with open(log_file, "w") as log:
    server = subprocess.Popen(
        [
            sys.executable,
            "-m", "uvicorn",
            "api:app",
            "--host", "127.0.0.1",
            "--port", "8001"
        ],
        cwd=str(project),
        stdout=log,
        stderr=subprocess.STDOUT
    )

time.sleep(4)

print("Server running:", server.poll() is None)
print("\nServer logs:")
print(log_file.read_text())

Server running: True

Server logs:



In [22]:
import requests

url = "http://127.0.0.1:8001"

response = requests.get(url, timeout=10)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 200
Response: {'message': 'Churn Prediction API is running'}


In [23]:
import requests

url = "http://127.0.0.1:8001/predict"

customer = {
    "usage": 5,
    "tenure": 8,
    "tickets": 3
}

response = requests.post(
    url,
    json=customer,
    timeout=10
)

print("Status Code:", response.status_code)
print("Prediction:", response.json())

Status Code: 200
Prediction: {'churn_probability': 0.7995, 'risk_level': 'High Churn Risk'}


In [24]:
import requests

url = "http://127.0.0.1:8001/predict"

invalid_customers = [
    {"usage": -5, "tenure": 8, "tickets": 3},
    {"usage": 5, "tickets": 3},
    {"usage": "unknown", "tenure": 8, "tickets": 3},
    {"usage": 5, "tenure": 8, "tickets": -1}
]

for customer in invalid_customers:
    response = requests.post(
        url,
        json=customer,
        timeout=10
    )

    print("Customer:", customer)
    print("Status Code:", response.status_code)
    print("-" * 40)

Customer: {'usage': -5, 'tenure': 8, 'tickets': 3}
Status Code: 422
----------------------------------------
Customer: {'usage': 5, 'tickets': 3}
Status Code: 422
----------------------------------------
Customer: {'usage': 'unknown', 'tenure': 8, 'tickets': 3}
Status Code: 422
----------------------------------------
Customer: {'usage': 5, 'tenure': 8, 'tickets': -1}
Status Code: 422
----------------------------------------


In [25]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8001,
    path="/docs",
    height=600
)

<IPython.core.display.Javascript object>

In [26]:
import requests

url = "http://127.0.0.1:8001/openapi.json"

response = requests.get(url, timeout=10)

print("Status Code:", response.status_code)

if response.status_code == 200:
    api_docs = response.json()

    print("API Name:", api_docs["info"]["title"])
    print("API Version:", api_docs["info"]["version"])

    print("\nAvailable Endpoints:")
    for path, methods in api_docs["paths"].items():
        for method in methods:
            print(method.upper(), path)

Status Code: 200
API Name: Customer Churn Prediction API
API Version: 1.0.0

Available Endpoints:
GET /
POST /predict


In [27]:
import subprocess
import sys
from pathlib import Path

project = Path("/content/churn-repo/customer-churn-app")

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd=str(project),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit Code:", result.returncode)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collecting ... collected 0 items

============================ no tests ran in 0.01s =============================

ERROR: file or directory not found: tests/


Exit Code: 4


In [28]:
from pathlib import Path

repo = Path("/content/churn-repo")
project = repo / "customer-churn-app"

print("Project exists:", project.exists())

print("\nExisting test files:")
test_files = list(repo.rglob("test_*.py"))

if test_files:
    for file in test_files:
        print(file.relative_to(repo))
else:
    print("No test files found")

Project exists: True

Existing test files:
customer-churn-app/Tests/test_prediction.py
customer-churn-app/Tests/test_api.py


In [29]:
from pathlib import Path
import shutil

project = Path("/content/churn-repo/customer-churn-app")

old_folder = project / "Tests"
new_folder = project / "tests"

new_folder.mkdir(exist_ok=True)

for filename in ["test_prediction.py", "test_api.py"]:
    source = old_folder / filename
    destination = new_folder / filename

    if source.exists():
        shutil.copy2(source, destination)
        print("Copied:", filename)

print("\nFinal test files:")
for file in new_folder.glob("test_*.py"):
    print(file.name)

Copied: test_prediction.py
Copied: test_api.py

Final test files:
test_prediction.py
test_api.py


In [30]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v"],
    cwd=str(project),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Exit Code:", result.returncode)

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/churn-repo/customer-churn-app
plugins: platformdirs-4.12.2, anyio-4.15.1, typeguard-4.6.0, langsmith-0.14.2
collecting ... collected 16 items

tests/test_api.py::test_home_endpoint PASSED                             [  6%]
tests/test_api.py::test_valid_prediction PASSED                          [ 12%]
tests/test_api.py::test_invalid_requests[customer0] PASSED               [ 18%]
tests/test_api.py::test_invalid_requests[customer1] PASSED               [ 25%]
tests/test_api.py::test_invalid_requests[customer2] PASSED               [ 31%]
tests/test_api.py::test_invalid_requests[customer3] PASSED               [ 37%]
tests/test_prediction.py::test_prediction_probability PASSED             [ 43%]
tests/test_prediction.py::test_customer_risk[5-8-3-High Churn Risk] PASSED [ 50%]
tests/test_

In [31]:
from google.colab import files

files.download(
    "/content/churn-repo/customer-churn-app/api.py"
)

files.download(
    "/content/churn-repo/customer-churn-app/requirements.txt"
)

files.download(
    "/content/churn-repo/customer-churn-app/tests/test_prediction.py"
)

files.download(
    "/content/churn-repo/customer-churn-app/tests/test_api.py"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>